# Challenge AI Engineer Intern Test

Notebook ini disusun agar siap dijalankan di Google Colab dan dapat langsung digunakan sebagai submission untuk tes AI Engineer Intern.

## Ringkasan Tugas

- **Bagian 1 yang dipilih:** Soal 1A - Object Tracking Mobil dengan pre-trained model YOLO
- **Bagian 2:** jawaban teori dasar AI secara ringkas, jelas, dan terstruktur

## Asumsi Implementasi

- Video input berisi setidaknya satu mobil yang terlihat cukup jelas oleh model pre-trained.
- Notebook ini menggunakan YOLOv5 pre-trained agar tidak memerlukan training ulang.
- Tracking yang digunakan adalah tracking sederhana berbasis centroid untuk menjaga notebook tetap ringan dan mudah dijalankan di Colab.


## Bagian 1A - Object Tracking Mobil dengan YOLO

Alur kerja notebook ini adalah sebagai berikut:

1. Menginstal library yang dibutuhkan.
2. Memuat model pre-trained YOLOv5.
3. Menyiapkan video input dari upload lokal.
4. Melakukan deteksi mobil pada setiap frame video.
5. Melakukan tracking antar-frame menggunakan centroid tracker.
6. Menyimpan hasil akhir ke file video output.


In [ ]:
# Install library yang dibutuhkan di Colab.
# torch biasanya sudah tersedia di Colab, tetapi OpenCV headless dan matplotlib kita pastikan ada.
!pip -q install opencv-python-headless matplotlib


In [ ]:
import cv2
import numpy as np
import torch
from collections import OrderedDict
from IPython.display import display, Video
from google.colab import files

print('CUDA tersedia:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


### 1) Memuat model YOLOv5 pre-trained

Pada bagian ini digunakan YOLOv5 pre-trained dari repository resmi Ultralytics. Model tersebut sudah dilatih pada dataset COCO, sehingga dapat mengenali kelas `car` tanpa training ulang.


In [ ]:
# Memuat YOLOv5s pre-trained.
# Model akan diunduh otomatis saat pertama kali dijalankan di Colab.
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
model = model.to(device)

# Pengaturan inferensi.
model.conf = 0.35
model.iou = 0.45

print('Model loaded.')
print('Nama kelas COCO untuk ID 2:', model.names[2])


### 2) Menyiapkan video input

Video input disiapkan melalui upload dari komputer lokal. Pendekatan ini dipilih karena paling stabil untuk penggunaan di Colab dan tidak bergantung pada ketersediaan URL eksternal.


In [ ]:
# Upload video dari komputer lokal.
# Setelah upload, pilih file video yang akan diproses.
uploaded = files.upload()
input_video_path = next(iter(uploaded.keys()))
print('Video input:', input_video_path)


### 3) Tracker sederhana berbasis centroid

YOLO menghasilkan bounding box pada setiap frame. Agar hasilnya juga menunjukkan identitas objek yang konsisten antar-frame, deteksi tersebut dihubungkan dengan tracker sederhana berbasis jarak centroid.

Logika dasarnya:

- Jika sebuah mobil muncul kembali pada lokasi yang mirip, objek tersebut akan mempertahankan ID yang sama.
- Jika objek tidak terdeteksi selama beberapa frame, ID tersebut akan dihapus.


In [ ]:
class CentroidTracker:
    def __init__(self, max_disappeared=15, max_distance=80):
        self.next_object_id = 0
        self.objects = OrderedDict()   # object_id -> bbox (x1, y1, x2, y2)
        self.centroids = OrderedDict() # object_id -> centroid (cx, cy)
        self.disappeared = OrderedDict()
        self.max_disappeared = max_disappeared
        self.max_distance = max_distance

    def _centroid(self, bbox):
        x1, y1, x2, y2 = bbox
        return np.array([(x1 + x2) // 2, (y1 + y2) // 2])

    def register(self, bbox):
        self.objects[self.next_object_id] = bbox
        self.centroids[self.next_object_id] = self._centroid(bbox)
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1

    def deregister(self, object_id):
        del self.objects[object_id]
        del self.centroids[object_id]
        del self.disappeared[object_id]

    def update(self, detections):
        # detections = list of bounding boxes [x1, y1, x2, y2]
        if len(detections) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.objects

        input_centroids = np.array([self._centroid(bbox) for bbox in detections])

        if len(self.objects) == 0:
            for bbox in detections:
                self.register(bbox)
            return self.objects

        object_ids = list(self.objects.keys())
        object_centroids = np.array(list(self.centroids.values()))

        # Hitung jarak centroid lama vs baru.
        distances = np.linalg.norm(object_centroids[:, None, :] - input_centroids[None, :, :], axis=2)

        rows = distances.min(axis=1).argsort()
        cols = distances.argmin(axis=1)[rows]

        used_rows = set()
        used_cols = set()

        for row, col in zip(rows, cols):
            if row in used_rows or col in used_cols:
                continue

            if distances[row, col] > self.max_distance:
                continue

            object_id = object_ids[row]
            self.objects[object_id] = detections[col]
            self.centroids[object_id] = input_centroids[col]
            self.disappeared[object_id] = 0

            used_rows.add(row)
            used_cols.add(col)

        unmatched_rows = set(range(len(object_ids))) - used_rows
        unmatched_cols = set(range(len(detections))) - used_cols

        for row in unmatched_rows:
            object_id = object_ids[row]
            self.disappeared[object_id] += 1
            if self.disappeared[object_id] > self.max_disappeared:
                self.deregister(object_id)

        for col in unmatched_cols:
            self.register(detections[col])

        return self.objects


### 4) Proses video: deteksi mobil + tracking + simpan hasil

Notebook ini hanya menampilkan kelas `car` agar fokus pada objek mobil. Jika ingin diperluas, kelas lain seperti `truck`, `bus`, atau `motorcycle` dapat ditambahkan sesuai kebutuhan.


In [ ]:
CAR_CLASS_ID = 2  # COCO: car

cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    raise RuntimeError('Video tidak bisa dibuka. Pastikan file video valid.')

fps = cap.get(cv2.CAP_PROP_FPS)
if fps is None or fps <= 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
output_video_path = 'cars_tracked_output.mp4'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

tracker = CentroidTracker(max_disappeared=15, max_distance=90)
frame_index = 0
total_cars_detected = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLOv5 menerima gambar RGB.
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = model(rgb_frame, size=640)
    detections = results.xyxy[0].cpu().numpy()

    car_boxes = []
    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        if int(cls) == CAR_CLASS_ID and conf >= model.conf:
            car_boxes.append([int(x1), int(y1), int(x2), int(y2)])

    tracked_objects = tracker.update(car_boxes)
    total_cars_detected += len(car_boxes)

    for object_id, bbox in tracked_objects.items():
        x1, y1, x2, y2 = bbox
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f'Car ID {object_id}',
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

    writer.write(frame)
    frame_index += 1

cap.release()
writer.release()

print('Jumlah frame diproses:', frame_index)
print('Total deteksi mobil mentah:', total_cars_detected)
print('Video hasil disimpan ke:', output_video_path)


### 5) Tampilkan hasil video

Jika runtime Colab menggunakan GPU, proses inferensi akan lebih cepat. Pada runtime CPU-only, proses tetap berjalan namun waktunya lebih lama.


In [ ]:
display(Video(output_video_path, embed=True, width=900))

# Jika ingin langsung mengunduh hasilnya ke laptop, aktifkan baris berikut.
# files.download(output_video_path)


## Bagian 2 - Teori AI

### Soal 2: Pertanyaan Teori Dasar

**a. Apa yang dimaksud dengan Artificial Intelligence (AI)? Sebutkan dua contohnya dalam kehidupan sehari-hari.**

Artificial Intelligence (AI) adalah bidang ilmu komputer yang membuat mesin mampu meniru kemampuan cerdas manusia, seperti mengenali pola, memahami bahasa, mengambil keputusan, dan belajar dari data.

Dua contoh AI dalam kehidupan sehari-hari:

- Rekomendasi video di YouTube atau Netflix.
- Asisten virtual seperti Siri, Google Assistant, atau ChatGPT.

**b. Apa perbedaan antara Supervised Learning dan Unsupervised Learning? Berikan satu contoh untuk masing-masing.**

- **Supervised Learning** menggunakan data yang sudah memiliki label jawaban. Model belajar dari pasangan input-output yang benar. Contoh: klasifikasi email spam dan bukan spam.
- **Unsupervised Learning** menggunakan data tanpa label. Model mencari pola atau struktur sendiri. Contoh: clustering pelanggan berdasarkan perilaku belanja.

### Soal 3: Pertanyaan Konsep

**a. Apa itu Feature dalam konteks machine learning? Mengapa penting untuk memilih fitur yang tepat saat membangun model?**

Feature adalah variabel atau atribut yang digunakan model sebagai masukan untuk mempelajari pola. Contohnya umur, pendapatan, atau jumlah klik.

Pemilihan fitur yang tepat penting karena fitur yang relevan membantu model belajar lebih akurat, lebih cepat, dan lebih stabil. Fitur yang kurang tepat dapat membuat model sulit belajar atau menghasilkan prediksi yang kurang baik.

**b. Apa itu Fine-tuning dalam machine learning? Sebutkan satu kasus di mana fine-tuning berguna.**

Fine-tuning adalah proses menyesuaikan model yang sudah pre-trained agar lebih cocok dengan tugas atau data yang lebih spesifik.

Contoh kasus yang berguna: model bahasa umum di-fine-tune untuk klasifikasi sentimen ulasan pelanggan pada domain e-commerce atau layanan keuangan.

### Kesimpulan

Notebook ini menunjukkan implementasi deteksi dan tracking mobil menggunakan YOLO pre-trained serta jawaban teori AI dasar dalam format yang siap dijalankan di Google Colab.
